# Complete results and figures

Generates every figure and number for the presentation, from your own data. Each figure is saved to
`./figures/` with a numbered filename, and the key numbers are written to `results.json` and
`results_tables.xlsx`.

Run top to bottom, then send me the `figures/` folder and `results.json` and I will build the deck.

| # | figure | what it shows |
|---|---|---|
| 01 | spectra and chemical bands | where coffee and chicory differ, and why |
| 02 | difference spectrum | the discriminating regions, isolated |
| 03 | preprocessing comparison | what each transform does to the spectra |
| 04 | session effect by preprocessing | why SNV fails and absorbance works |
| 05 | preprocessing selection | the accuracy-versus-transfer trade-off |
| 06 | latent variable selection | calibration vs cross-validation error — the overfitting curve |
| 07 | regression method comparison | PLSR against PCR, Ridge, SVR, Random Forest and others |
| 08 | final model fit | measured vs predicted, with residual diagnostics |
| 09 | regression coefficients and VIP | which wavelengths the model uses |
| 10 | external validation | the headline result |
| 11 | permutation test | that the model is not fitting chance |
| 12 | applicability domain | which samples the model may judge |
| 13 | unknown samples | A, B and C |

In [ ]:
import os
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.signal import savgol_filter
from scipy.stats import f as f_dist, chi2

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 11,
                     "figure.facecolor": "white", "axes.grid": True, "grid.alpha": .3,
                     "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

# ------------------------------- EDIT -------------------------------
BASE    = Path(os.environ.get("COFFEE_DATA", "../.."))
OLD_CSV = BASE / "samples_out" / "training_samples.csv"
NEW_CSV = BASE / "batch3_out"  / "training_samples.csv"
FIG     = Path("./figures")
WL_MIN, WL_MAX = 1000.0, 2450.0
TRIM_PCT, K_UNIT, N_COMP = 10, 8, 6
N_PERM = 300
# ---------------------------------------------------------------------

FIG.mkdir(exist_ok=True, parents=True)
RNG = np.random.default_rng(0)
RESULTS = {}

BANDS = [(1450, "O–H\nwater"), (1730, "C–H\nlipid"), (1940, "O–H\nwater"),
         (2100, "C–O\ncarbohydrate"), (2310, "C–H\ncombination")]
OLD_LAB = ["RB100", "RB[90:10]", "RB[80:20]", "RB[70:30]", "RB[60:40]", "RB[50:50]"]
NEW_LAB = ["RB[95:5]", "RB[85:15]", "RB[75:25]", "RB[65:35]", "RB[55:45]"]
CAL_LAB = OLD_LAB + NEW_LAB
print("old:", OLD_CSV.exists(), "| new:", NEW_CSV.exists())

In [ ]:
def load(p, tag=None):
    d = pd.read_csv(p); d = d[d.label != "AB100"].reset_index(drop=True)
    if tag:
        for c in ("scan", "group", "plate"): d[c] = d[c].astype(str) + tag
    return d

OLD, NEW = load(OLD_CSV), load(NEW_CSV, "_b3")
COMB = pd.concat([OLD, NEW], ignore_index=True)

wc = [c for c in COMB.columns if c.replace(".", "", 1).isdigit()]
wl_all = np.array([float(c) for c in wc])
BM = (wl_all >= WL_MIN) & (wl_all <= WL_MAX)
WL = wl_all[BM]; COLS = [c for c, k in zip(wc, BM) if k]
R_ALL = np.clip(COMB[COLS].values.astype(float), 1e-3, None)
R_FULL = np.clip(COMB[wc].values.astype(float), 1e-3, None)

print(f"{len(COMB)} patches | {COMB.group.nunique()} plates | {COMB.scan.nunique()} scans")
print(f"bands {len(WL)} ({WL[0]:.0f}-{WL[-1]:.0f} nm) of {len(wl_all)} available")
print(COMB.groupby(["day", "label"]).size().unstack(fill_value=0).to_string())
RESULTS["dataset"] = dict(patches=int(len(COMB)), plates=int(COMB.group.nunique()),
                          scans=int(COMB.scan.nunique()), bands=int(len(WL)),
                          levels=sorted(map(float, COMB.loc[COMB.role=="calibration","chicory_pct"].dropna().unique())))

## Figure 01 — Spectra and chemical bands

The physical basis of the whole method. Coffee carries 10–15 % lipid; chicory is inulin-rich and carries
almost none. That difference appears at the C–H first overtone near 1730 nm and the C–H combination band
near 2300 nm. Water dominates the spectrum at 1450 and 1940 nm but says little about composition.

In [ ]:
def annotate_bands(ax, ytop=None, fs=7.5):
    y = ytop if ytop is not None else ax.get_ylim()[1]
    for nm, lab in BANDS:
        ax.axvline(nm, color="0.6", lw=.9, ls=":", zorder=0)
        ax.text(nm, y, lab, fontsize=fs, rotation=90, va="top", ha="right", color="0.35")

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
pure = {"pure robusta (RB100)": "RB100", "50:50 robusta:chicory": "RB[50:50]"}
for lab, key in pure.items():
    s = (COMB.label == key).values
    if s.sum(): axs[0].plot(wl_all, R_FULL[s].mean(0), lw=2, label=lab)
axs[0].axvspan(wl_all[0], WL_MIN, color="crimson", alpha=.10)
axs[0].axvspan(WL_MAX, wl_all[-1], color="crimson", alpha=.10)
axs[0].set_xlabel("wavelength (nm)"); axs[0].set_ylabel("reflectance")
axs[0].set_title("SWIR reflectance — shaded regions excluded")
annotate_bands(axs[0]); axs[0].legend(loc="lower left")

cmap = plt.get_cmap("viridis")
cal = (COMB.role == "calibration").values
for lv in sorted(COMB.loc[cal, "chicory_pct"].unique()):
    s = cal & (COMB.chicory_pct.values == lv)
    axs[1].plot(WL, R_ALL[s].mean(0), lw=1.5, color=cmap(lv / 55), label=f"{lv:.0f} %")
axs[1].set_xlabel("wavelength (nm)"); axs[1].set_ylabel("reflectance")
axs[1].set_title("All 11 calibration levels (0–50 % chicory)")
annotate_bands(axs[1]); axs[1].legend(ncol=2, fontsize=8, title="chicory")
plt.tight_layout(); plt.savefig(FIG / "01_spectra_bands.png", bbox_inches="tight"); plt.show()

## Figure 02 — Where the discrimination actually comes from

Two independent views. The **difference spectrum** between the extremes shows where they part company; the
**correlation with chicory content** shows which bands track concentration across the whole series. Both
should peak in the same places, and they do.

In [ ]:
def snv(A): return (A - A.mean(-1, keepdims=True)) / (A.std(-1, keepdims=True) + 1e-12)

rb = snv(R_ALL[(COMB.label == "RB100").values]).mean(0)
ch = snv(R_ALL[(COMB.label == "CH100").values]).mean(0) if (COMB.label == "CH100").any() else None
Zabs = np.log10(1 / R_ALL); Zabs = Zabs - Zabs.mean(1, keepdims=True)
corr = np.array([np.corrcoef(Zabs[cal][:, k], COMB.chicory_pct.values[cal])[0, 1]
                 for k in range(WL.size)])

fig, axs = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True)
if ch is not None:
    axs[0].plot(WL, rb - ch, lw=1.8, color="purple")
    axs[0].fill_between(WL, 0, rb - ch, color="purple", alpha=.15)
axs[0].axhline(0, color="k", lw=.8); axs[0].set_ylabel("SNV difference")
axs[0].set_title("Pure robusta minus pure chicory")
axs[1].plot(WL, corr, lw=1.8, color="teal")
axs[1].fill_between(WL, 0, corr, color="teal", alpha=.15)
axs[1].axhline(0, color="k", lw=.8)
axs[1].set_ylabel("correlation with chicory %"); axs[1].set_xlabel("wavelength (nm)")
axs[1].set_title("Correlation of each band with chicory content")
for a in axs: annotate_bands(a)
plt.tight_layout(); plt.savefig(FIG / "02_discrimination.png", bbox_inches="tight"); plt.show()

k = int(np.argmax(np.abs(corr)))
print(f"strongest correlation: {WL[k]:.0f} nm, r = {corr[k]:+.3f}")
RESULTS["strongest_band_nm"] = float(WL[k]); RESULTS["strongest_band_r"] = float(corr[k])

## Figures 03–05 — Choosing the preprocessing

Your two calibration sessions differ by a uniform brightness factor of about 1.15, caused by packing
density and sample height rather than chemistry. Band depths — which are scale-free — are linear with
chicory content at r ≈ −0.99 across both sessions, so the material is consistent and only the brightness
scale moved.

That fact dictates the transform. In absorbance, `A = log10(1/R)`, a multiplicative factor becomes an
**additive offset**, and subtracting each spectrum's own mean removes it exactly. SNV also divides by the
standard deviation, which flatters within-session cross-validation but does not transfer.

The decisive test is not the CV score. It is: fit on one session, predict a sample of known composition
from the other.

In [ ]:
def msc(A, ref=None):
    # vectorised: solve every spectrum's slope and intercept in one matmul
    ref = A.mean(0) if ref is None else ref
    M = np.column_stack([ref, np.ones_like(ref)])
    coef = np.linalg.pinv(M) @ A.T                      # (2, n_spectra)
    m_, c_ = coef[0][:, None], coef[1][:, None]
    return (A - c_) / np.where(np.abs(m_) > 1e-9, m_, 1e-9)
def sg(A, d, w=15, p=2): return savgol_filter(A, w, p, deriv=d, axis=1)
absb   = lambda A: np.log10(1.0 / A)
offset = lambda A: A - A.mean(-1, keepdims=True)

PREP = {"raw": lambda A: A, "SNV": snv, "MSC": msc,
        "absorbance": absb,
        "absorbance+offset": lambda A: offset(absb(A)),
        "absorbance+SG1": lambda A: sg(absb(A), 1),
        "SG1": lambda A: sg(A, 1), "SG2": lambda A: sg(A, 2)}

fig, axs = plt.subplots(2, 4, figsize=(18, 7.4))
for ax, (nm, f) in zip(axs.ravel(), PREP.items()):
    Zp = f(R_ALL)
    for lv in sorted(COMB.loc[cal, "chicory_pct"].unique()):
        s = cal & (COMB.chicory_pct.values == lv)
        ax.plot(WL, Zp[s].mean(0), lw=1.1, color=cmap(lv / 55))
    ax.set_title(nm); ax.set_xlabel("nm")
    if nm == "absorbance+offset":
        ax.set_title(nm + "   ← chosen", fontweight="bold", color="darkgreen")
fig.suptitle("Figure 03 — preprocessing applied to the 11 calibration levels "
             "(dark = 0 %, light = 50 % chicory)", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "03_preprocessing.png", bbox_inches="tight"); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
for col, nm in enumerate(["raw", "SNV", "absorbance", "absorbance+offset"]):
    Zp = PREP[nm](R_ALL); T = PCA(n_components=2).fit_transform(Zp)
    for d, c_, mk in [(1, "tab:blue", "o"), (2, "tab:orange", "^"), (3, "tab:green", "s")]:
        s = (COMB.day == d).values
        if s.sum(): axs[0][col].scatter(T[s, 0], T[s, 1], s=5, alpha=.35, c=c_,
                                        marker=mk, label=f"session {d}")
    axs[0][col].set_title(f"{nm}\ncoloured by SESSION"); axs[0][col].legend(fontsize=8)
    sc = axs[1][col].scatter(T[cal, 0], T[cal, 1], s=5, c=COMB.chicory_pct.values[cal], cmap="viridis")
    axs[1][col].set_title("coloured by chicory %")
    for r_ in (0, 1): axs[r_][col].set_xlabel("PC1"); axs[r_][col].set_ylabel("PC2")
plt.colorbar(sc, ax=axs[1][3], label="chicory %")
fig.suptitle("Figure 04 — the session effect. Top row: sessions should overlap. "
             "Bottom row: the concentration gradient must survive.", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "04_session_effect.png", bbox_inches="tight"); plt.show()

In [ ]:
# transfer test: fit on session 2 only, predict session-1 pure robusta (truth 0 %)
s2 = ((COMB.day == 2) & COMB.chicory_pct.notna()).values
rb0 = (COMB.label == "RB100").values
y2 = COMB.chicory_pct.values[s2]
rows = []
for nm, f in PREP.items():
    Zp = f(R_ALL)
    for nc in (3, 4, 5, 6):
        p = np.full(s2.sum(), np.nan)
        for tr, te in LeaveOneGroupOut().split(Zp[s2], y2, groups=y2):
            p[te] = PLSRegression(n_components=nc).fit(Zp[s2][tr], y2[tr]).predict(Zp[s2][te]).ravel()
        m = PLSRegression(n_components=nc).fit(Zp[s2], y2)
        rows.append(dict(prep=nm, n_comp=nc,
                         rmsecv=float(np.sqrt(mean_squared_error(y2, p))),
                         session_bias=float(m.predict(Zp[rb0]).ravel().mean())))
TR = pd.DataFrame(rows)
TR.to_csv(FIG / "table_preprocessing.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
sub = TR[TR.n_comp == 5].sort_values("session_bias")
cols_ = ["seagreen" if abs(v) < 3 else "crimson" for v in sub.session_bias]
axs[0].barh(sub.prep, sub.session_bias, color=cols_)
axs[0].axvline(0, color="k", lw=1.4)
axs[0].set_xlabel("predicted chicory % for pure robusta (truth = 0)")
axs[0].set_title("Session transfer by preprocessing (5 LV)")
for _, r_ in TR.iterrows():
    good = r_.prep == "absorbance+offset"
    axs[1].scatter(r_.rmsecv, abs(r_.session_bias), s=60 if good else 34,
                   color="seagreen" if good else "0.65", zorder=3 if good else 2,
                   edgecolor="k" if good else "none", lw=.8)
best = TR.loc[TR.session_bias.abs().idxmin()]
axs[1].annotate("chosen", (best.rmsecv, abs(best.session_bias)), xytext=(18, 14),
                textcoords="offset points", fontsize=10, color="darkgreen", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="darkgreen"))
axs[1].set_xlabel("RMSECV within one session (%)"); axs[1].set_ylabel("|session bias| (%)")
axs[1].set_title("The trade-off: best CV is not best transfer")
plt.tight_layout(); plt.savefig(FIG / "05_preprocessing_choice.png", bbox_inches="tight"); plt.show()
print(TR.sort_values("session_bias", key=abs).head(6).round(2).to_string(index=False))
RESULTS["preprocessing"] = dict(chosen="absorbance+offset",
                                session_bias=round(float(best.session_bias), 2),
                                snv_bias=round(float(TR[(TR.prep=="SNV")&(TR.n_comp==5)].session_bias.iloc[0]), 2))

## Figure 06 — Choosing the number of latent variables

The classic overfitting picture. **RMSEC** (calibration error) falls monotonically as components are
added — it always does, because the model is fitting the same data it is scored on. **RMSECV**, from
leave-one-concentration-out, is the honest curve: it falls, flattens, then rises once the extra components
start describing noise instead of composition.

The third panel adds a constraint the usual picture leaves out: session transfer. A configuration that
scores well but fails to transfer is useless here, because every sample you want to predict comes from a
different session than most of the calibration.

In [ ]:
# build super-patches once, used everywhere below
def build_units(Z, meta, k):
    c_ = (meta.role == "calibration").values
    pre = PLSRegression(n_components=4).fit(Z[c_], meta.chicory_pct.values[c_])
    mu0 = Z[c_].mean(0); T0 = (Z - mu0) @ pre.x_rotations_
    q = np.sum(((Z - mu0) - T0 @ pre.x_loadings_.T) ** 2, 1)
    keep = pd.Series(q).groupby(meta.group.values).rank(pct=True).values <= (1 - TRIM_PCT / 100)
    idx = []
    for g_ in pd.unique(meta.group.values):
        ii = np.where((meta.group.values == g_) & keep)[0]
        if len(ii) < k: continue
        n = max(1, len(ii) // k)
        lab = KMeans(n_clusters=n, n_init=5, random_state=0).fit_predict(
            meta.loc[ii, ["patch_row", "patch_col"]].values.astype(float))
        idx += [ii[lab == c2] for c2 in range(n) if (lab == c2).sum() > 0]
    return (np.array([Z[i].mean(0) for i in idx]),
            meta.iloc[[i[0] for i in idx]].reset_index(drop=True))

Z_SEL = PREP["absorbance+offset"](R_ALL)
ZU, UM = build_units(Z_SEL, COMB, K_UNIT)
cu = UM.label.isin(CAL_LAB).values
Zc, Y, G = ZU[cu], UM.chicory_pct.values[cu], UM.group.values[cu]
print(f"{len(Y)} super-patches from {len(set(G))} plates, {len(set(Y))} levels")

comps = range(1, 11)
rmsec, rmsecv, bias = [], [], []
for nc in comps:
    m = PLSRegression(n_components=nc).fit(Zc, Y)
    rmsec.append(float(np.sqrt(mean_squared_error(Y, m.predict(Zc).ravel()))))
    p = np.full(len(Y), np.nan)
    for tr, te in LeaveOneGroupOut().split(Zc, Y, groups=Y):
        p[te] = PLSRegression(n_components=nc).fit(Zc[tr], Y[tr]).predict(Zc[te]).ravel()
    rmsecv.append(float(np.sqrt(mean_squared_error(Y, p))))
    m2 = PLSRegression(n_components=nc).fit(Z_SEL[s2], y2)
    bias.append(float(m2.predict(Z_SEL[rb0]).ravel().mean()))

fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
axs[0].plot(comps, rmsec, "o-", lw=2, label="RMSEC (calibration)")
axs[0].plot(comps, rmsecv, "s-", lw=2, label="RMSECV (leave-one-level-out)")
axs[0].axvline(N_COMP, color="crimson", ls="--", lw=1.6, label=f"chosen ({N_COMP} LV)")
axs[0].set_xlabel("latent variables"); axs[0].set_ylabel("RMSE (%)")
axs[0].set_title("Calibration vs validation error"); axs[0].legend()
axs[1].plot(comps, np.array(rmsecv) - np.array(rmsec), "o-", lw=2, color="tab:red")
axs[1].axhline(0, color="k", lw=.8); axs[1].axvline(N_COMP, color="crimson", ls="--", lw=1.6)
axs[1].set_xlabel("latent variables"); axs[1].set_ylabel("RMSECV − RMSEC (%)")
axs[1].set_title("Overfitting gap")
axs[2].plot(comps, bias, "o-", lw=2, color="tab:purple")
axs[2].axhline(0, color="k", lw=1); axs[2].axhspan(-3, 3, color="seagreen", alpha=.12)
axs[2].axvline(N_COMP, color="crimson", ls="--", lw=1.6)
axs[2].set_xlabel("latent variables"); axs[2].set_ylabel("session bias (%)")
axs[2].set_title("Transfer to the other session")
plt.tight_layout(); plt.savefig(FIG / "06_latent_variables.png", bbox_inches="tight"); plt.show()
RESULTS["lv_curve"] = dict(components=list(comps), rmsec=[round(v,3) for v in rmsec],
                           rmsecv=[round(v,3) for v in rmsecv], chosen=N_COMP)

## Figure 07 — Why PLS rather than anything else

Nine regression methods on identical data and identical splits. Two scores for each: leave-one-level-out
RMSECV, and session transfer.

The result is worth showing on a slide, because it is not the usual one. The flexible methods — Random
Forest, Gradient Boosting — do not win. With **11 distinct concentrations and 33 plates**, a flexible
model has enough freedom to recognise individual plates rather than learn composition, and the
leave-one-level-out split is what exposes that. Linear latent-variable methods are the right tool for
288 collinear predictors and a handful of levels.

In [ ]:
MODELS = {
    "PLSR (chosen)":     lambda: PLSRegression(n_components=N_COMP),
    "PLSR 3 LV":         lambda: PLSRegression(n_components=3),
    "PCR (PCA+OLS)":     lambda: make_pipeline(PCA(n_components=N_COMP), LinearRegression()),
    "Ridge":             lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "ElasticNet":        lambda: make_pipeline(StandardScaler(), ElasticNet(alpha=.01, l1_ratio=.5, max_iter=5000)),
    "SVR (linear)":      lambda: make_pipeline(StandardScaler(), SVR(kernel="linear", C=10)),
    "SVR (RBF)":         lambda: make_pipeline(StandardScaler(), SVR(kernel="rbf", C=100, gamma="scale")),
    "Random Forest":     lambda: RandomForestRegressor(n_estimators=120, random_state=0, n_jobs=-1),
    "Gradient Boosting": lambda: GradientBoostingRegressor(n_estimators=100, random_state=0),
}

def cv_rmse(mk, X, yv, groups):
    p = np.full(len(yv), np.nan)
    for tr, te in LeaveOneGroupOut().split(X, yv, groups=groups):
        p[te] = np.asarray(mk().fit(X[tr], yv[tr]).predict(X[te])).ravel()
    return float(np.sqrt(mean_squared_error(yv, p))), p

rows = []
for nm, mk in MODELS.items():
    r_lvl, p_lvl = cv_rmse(mk, Zc, Y, Y)
    fitted = mk().fit(Z_SEL[s2], y2)
    b = float(np.asarray(fitted.predict(Z_SEL[rb0])).ravel().mean())
    t = pd.DataFrame(dict(g=G, y=Y, p=p_lvl)).groupby(["g", "y"], as_index=False)["p"].mean()
    rows.append(dict(model=nm, rmsecv_level=r_lvl,
                     rmsecv_plate=float(np.sqrt(mean_squared_error(t.y, t.p))),
                     session_bias=b, r2=float(r2_score(Y, p_lvl))))
MC = pd.DataFrame(rows).sort_values("rmsecv_level")
MC.to_csv(FIG / "table_model_comparison.csv", index=False)
print(MC.round(2).to_string(index=False))

fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
o = MC.sort_values("rmsecv_level")
cols_ = ["seagreen" if m == "PLSR (chosen)" else "0.65" for m in o.model]
axs[0].barh(o.model, o.rmsecv_level, color=cols_)
axs[0].set_xlabel("RMSECV, leave-one-level-out (%)"); axs[0].set_title("Accuracy")
axs[0].invert_yaxis()
axs[1].barh(o.model, o.session_bias.abs(), color=cols_)
axs[1].axvline(3, color="crimson", ls="--", lw=1.2)
axs[1].set_xlabel("|session bias| (%)"); axs[1].set_title("Transfer to another session")
axs[1].invert_yaxis()
for _, r_ in MC.iterrows():
    good = r_.model == "PLSR (chosen)"
    axs[2].scatter(r_.rmsecv_level, abs(r_.session_bias), s=90 if good else 45,
                   color="seagreen" if good else "0.65", edgecolor="k" if good else "none",
                   lw=.8, zorder=3 if good else 2)
    axs[2].annotate(r_.model, (r_.rmsecv_level, abs(r_.session_bias)), fontsize=7,
                    xytext=(5, 4), textcoords="offset points")
axs[2].set_xlabel("RMSECV (%)"); axs[2].set_ylabel("|session bias| (%)")
axs[2].set_title("Both axes matter")
plt.tight_layout(); plt.savefig(FIG / "07_model_comparison.png", bbox_inches="tight"); plt.show()
RESULTS["model_comparison"] = MC.round(3).to_dict("records")

## Figures 08–09 — The fitted model

In [ ]:
MODEL = PLSRegression(n_components=N_COMP).fit(Zc, Y)
p_cv = np.full(len(Y), np.nan)
for tr, te in LeaveOneGroupOut().split(Zc, Y, groups=Y):
    p_cv[te] = PLSRegression(n_components=N_COMP).fit(Zc[tr], Y[tr]).predict(Zc[te]).ravel()
plate_t = pd.DataFrame(dict(g=G, y=Y, p=p_cv)).groupby(["g", "y"], as_index=False).agg(
    m=("p", "mean"), s=("p", "std"))
RMSECV_U = float(np.sqrt(mean_squared_error(Y, p_cv)))
RMSECV_P = float(np.sqrt(mean_squared_error(plate_t.y, plate_t.m)))
print(f"unit RMSECV {RMSECV_U:.2f} % | plate RMSECV {RMSECV_P:.2f} % | R2 {r2_score(Y, p_cv):.4f}")

fig, axs = plt.subplots(1, 3, figsize=(15.5, 4.5))
axs[0].scatter(Y + RNG.normal(0, .5, len(Y)), p_cv, s=10, alpha=.2, color="tab:blue", label="super-patches")
axs[0].errorbar(plate_t.y, plate_t.m, yerr=plate_t.s, fmt="o", color="crimson", capsize=4,
                ms=6, lw=1.2, label="plate mean ± sd")
axs[0].plot([-3, 55], [-3, 55], "k--", lw=1.2, label="1:1")
axs[0].set_xlim(-3, 55); axs[0].set_ylim(-3, 55)
axs[0].set_xlabel("true chicory (%)"); axs[0].set_ylabel("cross-validated prediction (%)")
axs[0].set_title(f"unit RMSECV {RMSECV_U:.2f} %  |  plate {RMSECV_P:.2f} %\nR² = {r2_score(Y, p_cv):.4f}")
axs[0].legend(loc="upper left")

lv = sorted(set(Y))
axs[1].bar(range(len(lv)), [np.mean(p_cv[Y == l] - l) for l in lv],
           color=["tab:orange" if l in (5, 15, 25, 35, 45) else "tab:blue" for l in lv])
axs[1].axhline(0, color="k", lw=1.2)
axs[1].set_xticks(range(len(lv)), [f"{l:.0f}" for l in lv])
axs[1].set_xlabel("chicory level (%)"); axs[1].set_ylabel("bias (%)")
axs[1].set_title("Bias per level  (orange = third session)")

res = p_cv - Y
axs[2].scatter(UM.patch_col.values[cu], res, s=8, alpha=.3)
z = np.polyfit(UM.patch_col.values[cu], res, 1)
xx = np.linspace(UM.patch_col.values[cu].min(), UM.patch_col.values[cu].max(), 40)
axs[2].plot(xx, np.poly1d(z)(xx), "r-", lw=1.8); axs[2].axhline(0, color="k", lw=1)
axs[2].set_xlabel("across-track position (pixel)"); axs[2].set_ylabel("residual (%)")
axs[2].set_title(f"Residual vs position: {z[0]*384:+.1f} % across the swath")
plt.tight_layout(); plt.savefig(FIG / "08_model_fit.png", bbox_inches="tight"); plt.show()
RESULTS["final_model"] = dict(n_latent=N_COMP, k_unit=K_UNIT, units=int(len(Y)),
                              plates=int(len(set(G))), levels=len(set(Y)),
                              rmsecv_unit=round(RMSECV_U, 2), rmsecv_plate=round(RMSECV_P, 2),
                              r2=round(float(r2_score(Y, p_cv)), 4))

In [ ]:
coef = MODEL.coef_.ravel()
if coef.size != WL.size: coef = MODEL.coef_.T.ravel()
Wt, Tt, Qt = MODEL.x_weights_, MODEL.x_scores_, MODEL.y_loadings_.ravel()
ss = (Qt ** 2) * (Tt ** 2).sum(0)
vip = np.sqrt(WL.size * ((Wt ** 2 / (Wt ** 2).sum(0)) @ ss) / ss.sum())

fig, axs = plt.subplots(2, 1, figsize=(11, 6.6), sharex=True)
axs[0].plot(WL, coef, lw=1.8, color="tab:blue")
axs[0].fill_between(WL, 0, coef, color="tab:blue", alpha=.15)
axs[0].axhline(0, color="k", lw=.9); axs[0].set_ylabel("PLS regression coefficient")
axs[0].set_title("What the model uses")
axs[1].plot(WL, vip, lw=1.8, color="darkorange")
axs[1].axhline(1, color="crimson", ls="--", lw=1.2, label="VIP = 1")
axs[1].fill_between(WL, 1, vip, where=vip > 1, color="darkorange", alpha=.2)
axs[1].set_ylabel("VIP"); axs[1].set_xlabel("wavelength (nm)"); axs[1].legend()
for a in axs: annotate_bands(a)
plt.tight_layout(); plt.savefig(FIG / "09_coefficients_vip.png", bbox_inches="tight"); plt.show()
top = np.sort(WL[np.argsort(-vip)[:12]]).astype(int)
print("12 most influential wavelengths (nm):", top)
RESULTS["top_wavelengths"] = [int(v) for v in top]

## Figure 10 — External validation

The strongest result here. A model trained on the first two sessions predicts the third-session samples
with no retraining and no correction. Independent preparations, a different scanning day, and a conversion
route the model had never seen.

In [ ]:
Zo = PREP["absorbance+offset"](np.clip(OLD[COLS].values.astype(float), 1e-3, None))
Zn = PREP["absorbance+offset"](np.clip(NEW[COLS].values.astype(float), 1e-3, None))
ZUo, UMo = build_units(Zo, OLD, 4)
co = UMo.label.isin(OLD_LAB).values
M_OLD = PLSRegression(n_components=5).fit(ZUo[co], UMo.chicory_pct.values[co])
pre_o = PLSRegression(n_components=4).fit(Zo[(OLD.role=="calibration").values],
                                          OLD.chicory_pct.values[(OLD.role=="calibration").values])
mu_o = Zo[(OLD.role=="calibration").values].mean(0)
Tn = (Zn - mu_o) @ pre_o.x_rotations_
qn = np.sum(((Zn - mu_o) - Tn @ pre_o.x_loadings_.T) ** 2, 1)
keepn = pd.Series(qn).groupby(NEW.group.values).rank(pct=True).values <= .9
idxn = []
for g_ in pd.unique(NEW.group.values):
    ii = np.where((NEW.group.values == g_) & keepn)[0]
    n = max(1, len(ii) // 4)
    lab = KMeans(n_clusters=n, n_init=5, random_state=0).fit_predict(
        NEW.loc[ii, ["patch_row", "patch_col"]].values.astype(float))
    idxn += [ii[lab == c2] for c2 in range(n) if (lab == c2).sum() > 0]
ZUn = np.array([Zn[i].mean(0) for i in idxn])
NMn = NEW.iloc[[i[0] for i in idxn]].reset_index(drop=True)
NMn["pred"] = M_OLD.predict(ZUn).ravel()

EXT = NMn.groupby(["label", "chicory_pct", "robusta_pct"], as_index=False).agg(
    predicted=("pred", "mean"), sd=("pred", "std"), n=("pred", "size")).sort_values("chicory_pct")
EXT["error"] = EXT.predicted - EXT.chicory_pct
EXT["robusta_pred"] = 100 - EXT.predicted
xv, yv = EXT.chicory_pct.values, EXT.predicted.values
SLOPE, INTER = np.polyfit(xv, yv, 1)
RMSEP = float(np.sqrt(np.mean((yv - xv) ** 2))); BIAS = float(np.mean(yv - xv))
sd_ref, rng_ = float(np.std(xv, ddof=1)), float(xv.max() - xv.min())
EXT.to_csv(FIG / "table_external_validation.csv", index=False)
print(EXT.round(2).to_string(index=False))
print(f"\nRMSEP {RMSEP:.2f} % | R2 {r2_score(xv, yv):.4f} | bias {BIAS:+.2f} | "
      f"slope {SLOPE:.3f} | intercept {INTER:+.2f}")
print(f"RPD {sd_ref/RMSEP:.1f} | RER {rng_/RMSEP:.1f}")

fig, axs = plt.subplots(1, 3, figsize=(15.5, 4.5))
axs[0].scatter(NMn.chicory_pct + RNG.normal(0, .4, len(NMn)), NMn.pred, s=11, alpha=.22, color="tab:blue")
axs[0].errorbar(EXT.chicory_pct, EXT.predicted, yerr=EXT.sd, fmt="o", color="crimson",
                capsize=5, ms=9, lw=1.6, zorder=5)
axs[0].plot([0, 52], [0, 52], "k--", lw=1.2); axs[0].set_xlim(0, 52); axs[0].set_ylim(0, 52)
axs[0].set_xlabel("true chicory (%)"); axs[0].set_ylabel("predicted chicory (%)")
axs[0].set_title(f"External validation\nRMSEP = {RMSEP:.2f} %,  R² = {r2_score(xv, yv):.4f}")
axs[1].bar(range(len(EXT)), EXT.error, color=["seagreen" if abs(e) < 2 else "tab:orange" for e in EXT.error])
axs[1].axhspan(-RMSEP, RMSEP, color="seagreen", alpha=.12); axs[1].axhline(0, color="k", lw=1.2)
axs[1].set_xticks(range(len(EXT)), EXT.label, rotation=35, ha="right")
axs[1].set_ylabel("error (%)"); axs[1].set_title("Error per sample")
w_ = .36; xp = np.arange(len(EXT))
axs[2].bar(xp - w_/2, EXT.robusta_pct, w_, label="true", color="tab:blue")
axs[2].bar(xp + w_/2, EXT.robusta_pred, w_, label="predicted", color="tab:orange")
for i, v in enumerate(EXT.robusta_pred): axs[2].text(i, v + 2, f"{v:.1f}", ha="center", fontsize=9)
axs[2].set_xticks(xp, EXT.label, rotation=35, ha="right"); axs[2].set_ylim(0, 110)
axs[2].set_ylabel("robusta content (%)"); axs[2].set_title("Robusta content"); axs[2].legend()
plt.tight_layout(); plt.savefig(FIG / "10_external_validation.png", bbox_inches="tight"); plt.show()
RESULTS["external_validation"] = dict(rmsep=round(RMSEP,2), r2=round(float(r2_score(xv,yv)),4),
    bias=round(BIAS,2), slope=round(float(SLOPE),3), intercept=round(float(INTER),2),
    rpd=round(sd_ref/RMSEP,2), rer=round(rng_/RMSEP,2), n_samples=int(len(EXT)))

## Figures 11–13 — Significance, domain, and the unknowns

The **permutation test** asks whether a good score could arise by chance. With 11 levels the model might
simply be separating objects and happening to order them; shuffling the concentrations across whole plates
and refitting gives the null distribution for exactly that.

The **applicability domain** decides whether the model may judge a sample at all. Q measures distance off
the model plane, scaled against pure robusta so the session component cancels.

In [ ]:
py_ = pd.DataFrame(dict(g=G, y=Y)).groupby("g")["y"].first()
null = []
for _ in range(N_PERM):
    mp = dict(zip(py_.index, RNG.permutation(py_.values)))
    yp = np.array([mp[g_] for g_ in G])
    p = np.full(len(yp), np.nan)
    for tr, te in LeaveOneGroupOut().split(Zc, yp, groups=yp):
        p[te] = PLSRegression(n_components=N_COMP).fit(Zc[tr], yp[tr]).predict(Zc[te]).ravel()
    null.append(np.sqrt(mean_squared_error(yp, p)))
null = np.array(null)
PVAL = float((np.sum(null <= RMSECV_U) + 1) / (len(null) + 1))

MU_C = Zc.mean(0)
def Qf(S):
    S = np.atleast_2d(S); T = (S - MU_C) @ MODEL.x_rotations_
    return np.sum(((S - MU_C) - T @ MODEL.x_loadings_.T) ** 2, 1)
QREF = float(np.median(Qf(ZU[(UM.label == "RB100").values])))

fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
axs[0].hist(null, bins=30, color="0.7", edgecolor="w", label="shuffled labels")
axs[0].axvline(RMSECV_U, color="crimson", lw=2.5, label=f"real model ({RMSECV_U:.2f} %)")
axs[0].set_xlabel("RMSECV (%)"); axs[0].set_ylabel("count"); axs[0].legend()
axs[0].set_title(f"Permutation test (n = {N_PERM})\np = {PVAL:.4f}")

qr = {}
for l in CAL_LAB + ["A", "B", "C", "CH100"]:
    s = (UM.label == l).values
    if s.sum(): qr[l] = float(np.median(Qf(ZU[s])) / QREF)
qs = pd.Series(qr).sort_values()
axs[1].barh(qs.index, qs.values, color=["seagreen" if v < 2 else "crimson" for v in qs.values])
axs[1].axvline(1, color="k", lw=1.2); axs[1].axvline(2, color="crimson", ls="--", lw=1.2)
axs[1].set_xlabel("median Q ÷ pure robusta Q"); axs[1].set_title("Applicability domain")
axs[1].tick_params(labelsize=8)

rows = []
for l in ["A", "B", "C"]:
    s = (UM.label == l).values
    if s.sum() == 0: continue
    pr = MODEL.predict(ZU[s]).ravel(); q_ = qr.get(l, np.nan)
    ok = q_ < 2 and -5 <= pr.mean() <= 55
    rows.append(dict(sample=l, chicory=round(float(pr.mean()), 1),
                     robusta=round(100 - float(pr.mean()), 1) if ok else np.nan,
                     unc=round(float(np.sqrt(RMSECV_U**2 + pr.std()**2)), 1),
                     Q_rel=round(q_, 2), verdict="quantifiable" if ok else "detect only"))
UNK = pd.DataFrame(rows)
if len(UNK):
    axs[2].errorbar(range(len(UNK)), UNK.chicory, yerr=UNK.unc, fmt="o", capsize=6, ms=10, color="tab:blue")
    for i, r_ in enumerate(UNK.itertuples()):
        ok = r_.verdict == "quantifiable"
        axs[2].annotate(f"{r_.chicory:.1f} ± {r_.unc:.1f}" + ("" if ok else "\n(detect only)"),
                        (i, r_.chicory), xytext=(14, -4), textcoords="offset points",
                        fontsize=9, color="k" if ok else "crimson")
    axs[2].axhspan(0, 50, color="seagreen", alpha=.10, label="calibrated range")
    axs[2].set_xticks(range(len(UNK)), UNK["sample"]); axs[2].set_ylim(0, 100)
    axs[2].set_xlim(-.5, len(UNK) - .1); axs[2].set_ylabel("chicory (%)")
    axs[2].set_title("Unknown commercial samples"); axs[2].legend(loc="upper left")
plt.tight_layout(); plt.savefig(FIG / "11_significance_domain_unknowns.png", bbox_inches="tight"); plt.show()
print(UNK.to_string(index=False))
print(f"\npermutation p = {PVAL:.4f}   (null centred at {null.mean():.1f} %)")
RESULTS["permutation_p"] = PVAL
RESULTS["unknowns"] = UNK.to_dict("records")

## Summary — everything in one place

In [ ]:
LIT = pd.DataFrame([
    dict(study="Harohally 2021 — robusta+chicory, FT-NIR", range_pct="0–100", levels="~40", LV=5,  error=1.29, spatial="no"),
    dict(study="Harohally 2021 — arabica+chicory, FT-NIR", range_pct="0–100", levels="~40", LV=3,  error=0.90, spatial="no"),
    dict(study="Forchetti 2020 — NIR-HSI + MCR",           range_pct="1–40",  levels="5",   LV=2,  error=4.00, spatial="4 mm²"),
    dict(study="Coqueiro 2024 — NIR, coffee husk",         range_pct="0–50",  levels="6",   LV=20, error=3.94, spatial="no"),
    dict(study="THIS WORK — SWIR-HSI (external)",          range_pct="0–50",  levels="11",  LV=N_COMP, error=RMSEP, spatial="full dish"),
])
LIT.to_csv(FIG / "table_literature.csv", index=False)
print(LIT.round(2).to_string(index=False))

RESULTS["literature"] = LIT.round(3).to_dict("records")
(Path("results.json")).write_text(json.dumps(RESULTS, indent=2, default=float))
with pd.ExcelWriter("results_tables.xlsx") as xl:
    TR.round(3).to_excel(xl, sheet_name="preprocessing", index=False)
    MC.round(3).to_excel(xl, sheet_name="model_comparison", index=False)
    EXT.round(3).to_excel(xl, sheet_name="external_validation", index=False)
    LIT.round(3).to_excel(xl, sheet_name="literature", index=False)
    if len(UNK): UNK.to_excel(xl, sheet_name="unknowns", index=False)

print("\n" + "=" * 70)
print("HEADLINE NUMBERS")
print("=" * 70)
print(f"  external RMSEP        {RMSEP:.2f} %   (R² {r2_score(xv, yv):.4f}, bias {BIAS:+.2f})")
print(f"  RPD / RER             {sd_ref/RMSEP:.1f} / {rng_/RMSEP:.1f}")
print(f"  plate RMSECV          {RMSECV_P:.2f} %   ({len(set(Y))} levels, {len(set(G))} plates)")
print(f"  permutation p         {PVAL:.4f}")
print(f"  strongest band        {RESULTS['strongest_band_nm']:.0f} nm (r = {RESULTS['strongest_band_r']:+.2f})")
print(f"  preprocessing         absorbance + offset removal")
print(f"  session bias          {RESULTS['preprocessing']['session_bias']:+.2f} %  "
      f"(SNV would be {RESULTS['preprocessing']['snv_bias']:+.2f} %)")
print(f"  validity              0–50 % chicory  =  50–100 % robusta")
print("=" * 70)
print(f"\n{len(list(FIG.glob('*.png')))} figures in {FIG.resolve()}")
for f in sorted(FIG.glob("*.png")): print("   ", f.name)
print("\nsend me the figures folder + results.json and I will build the deck")